In [ ]:
import json
import numpy as np
import heapq
from openai import OpenAI


def batch_cosine_similarity(
    query: list[float], documents: list[list[float]]
) -> list[float]:
    """
    使用NumPy高效批量计算余弦相似度

    Args:
        query: 查询向量
        documents: 文档向量列表

    Returns:
        相似度列表
    """
    # 转换为numpy数组
    query_arr = np.array(query)
    docs_arr = np.array(documents)  # shape: (n_docs, embedding_dim)

    # 向量化计算点积：docs_arr @ query_arr 得到所有文档与query的点积
    dot_products = docs_arr @ query_arr  # shape: (n_docs,)

    # 计算范数
    query_norm = np.linalg.norm(query_arr)
    docs_norms = np.linalg.norm(docs_arr, axis=1)  # shape: (n_docs,)

    # 计算余弦相似度
    similarities = dot_products / (query_norm * docs_norms)

    # 转换回Python list
    return similarities.tolist()


def remember(memory: str) -> None:
    """
    记住一条记忆
    """
    d = {"content": memory, "embedding": get_embedding(memory)}
    MEMORY_BANK.append(d)
    EMBEDDINGS.append(d["embedding"])


OPENAI_CLIENT = OpenAI(
    api_key="EMPTY",
    base_url=f"http://127.0.0.1:9002/v1",
)


def get_embedding(text: str) -> list[float]:
    """
    获取文本的嵌入向量
    """  # see: https://github.com/modelscope/ms-swift/blob/main/examples/deploy/embedding/client.py
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": text},
            ],
        }
    ]
    response = OPENAI_CLIENT.chat.completions.create(
        model="Qwen3-Embedding-0.6B",
        messages=messages,
        temperature=0.0,
    )
    emb = response.data[0]["embedding"]
    emb = emb / np.linalg.norm(emb)  # 归一化
    return emb.tolist()


def search(query: str, top_k: int = 5) -> list[str] | None:
    """
    根据查询字符串搜索记忆
    """
    # 计算查询向量的嵌入
    query_embedding = get_embedding(query)

    # 计算所有记忆与查询的相似度
    sims = batch_cosine_similarity(query_embedding, EMBEDDINGS)

    # 使用heapq.nlargest只选出top_k，时间复杂度O(n log k)
    top_k_indices = heapq.nlargest(top_k, range(len(sims)), key=lambda i: sims[i])

    # 返回top_k条记忆
    return [MEMORY_BANK[i]["content"] for i in top_k_indices]


MEMORY_BANK: list[dict[str,]] = []
EMBEDDINGS: list[list[float]] = []
MEMORY_BANK_FILE = "memory_bank.json"


def reload_memory() -> None:
    """
    加载记忆
    """
    global MEMORY_BANK, EMBEDDINGS
    with open(MEMORY_BANK_FILE, "r", encoding="utf-8") as f:
        MEMORY_BANK = json.load(f)
    EMBEDDINGS = [d["embedding"] for d in MEMORY_BANK]


def save_memory() -> None:
    """
    保存记忆
    """
    with open(MEMORY_BANK_FILE, "w", encoding="utf-8") as f:
        json.dump(MEMORY_BANK, f, ensure_ascii=False, indent=2)

In [ ]:
remember("你好")

In [ ]:
remember("吃什么？")

In [ ]:
search("你好?")

In [ ]:
search("hello")

In [ ]:
r=get_embedding("你好")

In [ ]:
np.linalg.norm(r)

In [ ]:
import requests

#remember
data={"content": "你好"}
response = requests.post("http://127.0.0.1:9003/remember", json=data)

#search
data={"query": "你好", "top_k": 1}
response = requests.post("http://127.0.0.1:9003/search", json=data)
print(response.json())# {"memory": ["你好"]}

#search
data={"query": "你好"}
response = requests.post("http://127.0.0.1:9003/search", json=data)
print(response.json())# {"memory": ["你好","hello","hi","你好吗?","你还好吗?"]}
